# RealLift: End-to-End Geographic Experiment Tutorial

Welcome to the **RealLift Quick Start Guide**. In this tutorial, we will walk through the complete lifecycle of a geographic incremental test, from generating historical data to selecting the optimal test clusters and measuring the final business impact.

RealLift uses **Synthetic Control** and **Matched Difference-in-Differences (DiD)** methodologies to provide a robust, counterfactual-based framework for marketing measurement.

## 1. Installation

First, let's install the library from PyPI.

In [ ]:
%pip install reallift

## 2. Step 1: Historical Data Simulation (Pre-test)

Before running an experiment, we need historical baseline data to understand natural correlations between geographies. 

We will use `generate_geo_data` to create a dataset for 27 geographies with:
- **Seasonality**: Simulating weekly patterns (period=7).
- **Trend**: General market growth.
- **Noise**: Random variance to test robustness.
- **Pre-only**: We will generate only the baseline period (90 days).

In [ ]:
from reallift import generate_geo_data

file_pre_test = 'demo_geodata_pre_test.csv'
start_date = "2025-01-01"
end_date = "2025-03-31"

geo_data = generate_geo_data(
    start_date = start_date,
    end_date = end_date,
    n_geos=27,
    pre_only=True,
    trend_slope=0.01,
    seasonality_amplitude=3,
    seasonality_period=7,
    noise_std=[1, 1.5],
    base_value=[50, 100],
    random_seed=42,
    save_csv=True,
    pre_file_name=file_pre_test,
    as_integer=True,
)

## 3. Step 2: Design of Experiments (DoE)

The **Design of Experiments (DoE)** phase is critical. It evaluates which combinations of treatment and control geos provide the highest statistical power and lowest prediction error.

The output will show:
- **Cluster Recommendations**: Which geos should be treated.
- **MDE (Minimum Detectable Effect)**: What is the smallest lift we can confidently detect for a given duration.

In [ ]:
from reallift import design_of_experiments

doe = design_of_experiments(
    filepath=file_pre_test,
    date_col="date",
    start_date=start_date,
    end_date=end_date,
    pct_treatment=None,
    fixed_treatment=None,
    experiment_days=[21, 28, 30, 35, 60]
)

## 4. Step 3: Injecting Simulated Marketing Intervention

To validate that our model can actually "find" an effect, we will take the optimal treatment geographies from the DoE (`treatment_pool`) and manually inject an artificial lift of 5% to 10% for 21 days.

In [ ]:
from reallift import generate_simulated_intervention

file_post_test = 'demo_geodata_post_test.csv'

geo_data_intervention = generate_simulated_intervention(
    filepath=file_pre_test,
    days=21,
    treatment_geos=doe['scenarios'][1]['treatment_pool'], # Using the best scenario from DoE
    lift=[0.05, 0.10],
    trend_slope=0.01,
    seasonality_amplitude=3,
    seasonality_period=7,
    noise_std=[1, 1.5],
    random_seed=42,
    save_csv=True,
    as_integer=True,
    file_name=file_post_test
)

## 5. Step 4: Measuring the Incremental Impact

Now we run the inference pipeline. This step inherits the experiment design from the DoE (`scenario=1`) and calculates:
- **Total Incremental Lift**: The count/financial gain caused by the campaign.
- **Bootstrap Significance**: Confirming the lift is statistically different from zero.
- **Placebo MSPE Ratio**: A rigorous test to ensure the effect is unique to our treated areas.

In [ ]:
from reallift import run_geo_experiment

results = run_geo_experiment(
    filepath=file_post_test,
    date_col="date",
    treatment_start_date="2025-04-01",
    treatment_end_date="2025-04-22",
    doe=doe, 
    scenario=1, # Matching the scenario choice in Step 3
    plot=True
)